# AI Job Market — Exploratory Data Analysis

This notebook explores the `ai_jobs_global.csv` dataset collected via the
Adzuna and USAJobs public APIs.

**Sections**
1. Dataset Overview
2. Top 10 Countries by Job Count
3. Salary Distribution by Country
4. Top 20 Most In-Demand Skills
5. Remote vs Hybrid vs Onsite
6. Experience Level Distribution
7. Job Postings Trend Over Time
8. Salary vs Experience Level

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
from collections import Counter

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Consistent colour palette across all charts
PALETTE = px.colors.qualitative.Bold

print('Libraries loaded ✓')

In [ ]:
# ── Load dataset ─────────────────────────────────────────────────────────────
CSV_PATH = 'data/ai_jobs_global.csv'

df = pd.read_csv(CSV_PATH, parse_dates=['posted_date'])
print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head(3)

---
## 1. Dataset Overview

In [ ]:
# ── 1. Dataset Overview ───────────────────────────────────────────────────────
print('=' * 55)
print(f'  Shape            : {df.shape[0]:,} rows  ×  {df.shape[1]} columns')
print(f'  Date range       : {df["posted_date"].min()}  →  {df["posted_date"].max()}')
print(f'  Unique countries : {df["country"].nunique()}')
print(f'  Unique companies : {df["company"].nunique():,}')
print('=' * 55)

# Data types + null counts
overview = pd.DataFrame({
    'dtype'      : df.dtypes,
    'non_null'   : df.notna().sum(),
    'null_count' : df.isna().sum(),
    'null_%'     : (df.isna().mean() * 100).round(1),
    'unique'     : df.nunique(),
})
display(overview)

---
## 2. Top 10 Countries by Job Count

In [ ]:
# ── 2. Top 10 Countries by Job Count ─────────────────────────────────────────
country_counts = (
    df['country']
    .value_counts()
    .head(10)
    .reset_index()
    .rename(columns={'index': 'country', 'country': 'job_count'})
)

# Ensure correct column names regardless of pandas version
country_counts.columns = ['country', 'job_count']

fig = px.bar(
    country_counts,
    x='country',
    y='job_count',
    color='country',
    color_discrete_sequence=PALETTE,
    title='Top 10 Countries by Number of AI Job Postings',
    labels={'country': 'Country', 'job_count': 'Number of Jobs'},
    text='job_count',
)
fig.update_traces(textposition='outside')
fig.update_layout(
    showlegend=False,
    plot_bgcolor='white',
    yaxis=dict(gridcolor='#eeeeee'),
    title_font_size=18,
)
fig.show()

---
## 3. Salary Distribution by Country

In [ ]:
# ── 3. Salary Distribution by Country (Box Plot) ─────────────────────────────
salary_df = df[df['salary_min'].notna()].copy()

# Keep top countries for readability
top_countries = salary_df['country'].value_counts().head(8).index.tolist()
salary_df = salary_df[salary_df['country'].isin(top_countries)]

fig = px.box(
    salary_df,
    x='country',
    y='salary_min',
    color='country',
    color_discrete_sequence=PALETTE,
    title='Annual Salary Distribution by Country (USD, Min Reported)',
    labels={'salary_min': 'Salary Min (USD)', 'country': 'Country'},
    points='outliers',
)
fig.update_layout(
    showlegend=False,
    plot_bgcolor='white',
    yaxis=dict(gridcolor='#eeeeee', tickprefix='$', tickformat=',.0f'),
    title_font_size=18,
)
fig.show()

print(f'\nRows with salary data: {len(salary_df):,} / {len(df):,} '
      f'({len(salary_df)/len(df)*100:.1f}%)')

---
## 4. Top 20 Most In-Demand Skills

In [ ]:
# ── 4. Top 20 Most In-Demand Skills (Horizontal Bar) ─────────────────────────
skill_series = df['required_skills'].dropna()
skill_series = skill_series[skill_series != '']

# Flatten comma-separated skill strings
all_skills: list[str] = []
for entry in skill_series:
    all_skills.extend([s.strip() for s in str(entry).split(',') if s.strip()])

skill_counts = (
    pd.Series(Counter(all_skills))
    .sort_values(ascending=True)
    .tail(20)
    .reset_index()
)
skill_counts.columns = ['skill', 'count']

fig = px.bar(
    skill_counts,
    x='count',
    y='skill',
    orientation='h',
    color='count',
    color_continuous_scale='Blues',
    title='Top 20 Most In-Demand AI/ML Skills',
    labels={'count': 'Number of Job Postings', 'skill': 'Skill'},
    text='count',
)
fig.update_traces(textposition='outside')
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#eeeeee'),
    coloraxis_showscale=False,
    title_font_size=18,
    height=600,
)
fig.show()

---
## 5. Remote vs Hybrid vs Onsite

In [ ]:
# ── 5. Remote / Hybrid / Onsite Breakdown (Pie) ───────────────────────────────
remote_counts = df['remote_type'].value_counts().reset_index()
remote_counts.columns = ['remote_type', 'count']

fig = px.pie(
    remote_counts,
    names='remote_type',
    values='count',
    color_discrete_sequence=PALETTE,
    title='Work Arrangement Breakdown',
    hole=0,
)
fig.update_traces(
    textposition='inside',
    textinfo='percent+label',
    pull=[0.03] * len(remote_counts),
)
fig.update_layout(title_font_size=18)
fig.show()

---
## 6. Experience Level Distribution

In [ ]:
# ── 6. Experience Level Distribution (Donut) ──────────────────────────────────
exp_order = ['Junior', 'Mid-level', 'Senior', 'Lead', 'Management']
exp_counts = (
    df['experience_level']
    .value_counts()
    .reindex(exp_order)
    .dropna()
    .reset_index()
)
exp_counts.columns = ['experience_level', 'count']

fig = px.pie(
    exp_counts,
    names='experience_level',
    values='count',
    color_discrete_sequence=PALETTE,
    title='Experience Level Distribution of AI Job Postings',
    hole=0.45,
)
fig.update_traces(
    textposition='outside',
    textinfo='percent+label',
)
fig.add_annotation(
    text=f'<b>{len(df):,}</b><br>Jobs',
    x=0.5, y=0.5,
    font_size=16,
    showarrow=False,
)
fig.update_layout(title_font_size=18, showlegend=True)
fig.show()

---
## 7. Job Postings Trend Over Time

In [ ]:
# ── 7. Job Postings Trend Over Time (Line Chart) ──────────────────────────────
trend_df = df[df['posted_date'].notna()].copy()
trend_df['posted_date'] = pd.to_datetime(trend_df['posted_date'], errors='coerce')
trend_df = trend_df.dropna(subset=['posted_date'])

# Aggregate by week to smooth out noise
trend_df['week'] = trend_df['posted_date'].dt.to_period('W').dt.start_time
weekly_counts = (
    trend_df.groupby(['week', 'source'])
    .size()
    .reset_index(name='job_count')
)

fig = px.line(
    weekly_counts,
    x='week',
    y='job_count',
    color='source',
    color_discrete_sequence=PALETTE,
    markers=True,
    title='Weekly AI Job Postings Trend by Source',
    labels={'week': 'Week', 'job_count': 'Number of Jobs', 'source': 'Source'},
)
fig.update_layout(
    plot_bgcolor='white',
    yaxis=dict(gridcolor='#eeeeee'),
    xaxis=dict(gridcolor='#eeeeee'),
    title_font_size=18,
)
fig.show()

---
## 8. Salary vs Experience Level

In [ ]:
# ── 8. Salary vs Experience Level (Violin Plot) ────────────────────────────────
violin_df = df[df['salary_min'].notna()].copy()

# Remove extreme outliers (above 99th percentile)
p99 = violin_df['salary_min'].quantile(0.99)
violin_df = violin_df[violin_df['salary_min'] <= p99]

exp_order = ['Junior', 'Mid-level', 'Senior', 'Lead', 'Management']
violin_df['experience_level'] = pd.Categorical(
    violin_df['experience_level'], categories=exp_order, ordered=True
)
violin_df = violin_df.sort_values('experience_level')

fig = px.violin(
    violin_df,
    x='experience_level',
    y='salary_min',
    color='experience_level',
    color_discrete_sequence=PALETTE,
    box=True,
    points='outliers',
    title='Annual Salary Distribution by Experience Level (USD)',
    labels={
        'salary_min': 'Min Salary (USD)',
        'experience_level': 'Experience Level',
    },
)
fig.update_layout(
    showlegend=False,
    plot_bgcolor='white',
    yaxis=dict(gridcolor='#eeeeee', tickprefix='$', tickformat=',.0f'),
    title_font_size=18,
)
fig.show()

# Summary stats table
stats = (
    violin_df
    .groupby('experience_level', observed=True)['salary_min']
    .describe(percentiles=[0.25, 0.5, 0.75])
    .round(0)
    .astype(int)
)
display(stats)

---
## Bonus: Top Job Titles

In [ ]:
# ── Bonus: Most Common Job Titles ─────────────────────────────────────────────
title_counts = (
    df['job_title']
    .value_counts()
    .head(15)
    .sort_values(ascending=True)
    .reset_index()
)
title_counts.columns = ['job_title', 'count']

fig = px.bar(
    title_counts,
    x='count',
    y='job_title',
    orientation='h',
    color='count',
    color_continuous_scale='Teal',
    title='Top 15 Most Frequent Job Titles',
    labels={'count': 'Count', 'job_title': 'Job Title'},
    text='count',
)
fig.update_traces(textposition='outside')
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#eeeeee'),
    coloraxis_showscale=False,
    title_font_size=18,
    height=500,
)
fig.show()